# Prototype


In [13]:
import importlib
import evaluate as evaluate_module
import analysis as analysis_module
from evaluate import evaluate, print_metrics, print_evaluation_summary
from benchmarks import load_benchmarks
from llm import call_llm, build_prompt
from analysis import analyze
import output as output_module

Loading of the benchmarks.


In [ ]:
# Configuration: change these variables as needed
from pathlib import Path
BENCHMARK_DIR = Path("./benchmark")
BENCHMARK_TYPES = ["socbenchd_1"]  # add more benchmark types here
BENCHMARK_LIMIT = None  # set to None for all sectors (these are e.g. socbenchd_1)
QUERY_LIMIT = 50        # set to None for all queries (applies across all sectors combined)
BASELINE = True        # True = skip refinement LLM call (first cycle only)
MAX_WORKERS = 10        # number of parallel LLM calls
MODEL = "deepseek-ai/DeepSeek-V4-Pro"

benchmark_sets, total_available, total_queries_available = load_benchmarks(BENCHMARK_DIR, BENCHMARK_TYPES, BENCHMARK_LIMIT, None)
total_queries = min(QUERY_LIMIT, total_queries_available) if QUERY_LIMIT is not None else total_queries_available
print(f"Loaded {len(benchmark_sets)} benchmark sets (total available: {total_available})")
print(f"Total queries: {total_queries} (of {total_queries_available} available)")
print(f"Mode: {'BASELINE' if BASELINE else 'REFINED'} | Workers: {MAX_WORKERS} | Model: {MODEL}")

Loaded 11 benchmark sets (total available: 11)
Total queries: 50 (of 110 available)
Mode: BASELINE | Workers: 10 | Model: deepseek-ai/DeepSeek-V4-Pro


In [15]:
# Editable prompt template: modify this cell to change the instruction given to the LLM.
PROMPT_TEMPLATE = '''You're doing a Service Composition.
You are given a set of REST API specifications and a task description.
Your job is to write Python code using the appropriate client library that fulfills the task by calling the necessary endpoints in the correct order. Import requests and create a function called compose.

Rules:
- Use the requests library.
- Only use endpoints defined in the provided specifications.
- Return ONLY raw Python code. No markdown, no code fences, no comments, no notes, no explanations — nothing but the code itself.
- Import the requests library and create a function called compose, where all the requests shall be called. Do NOT call that function.

source
{services_block}

## Task

{query}

## Python Code
'''


In [16]:
# Call the LLM for all benchmark queries in parallel
from concurrent.futures import ThreadPoolExecutor, as_completed

# Collect tasks up to QUERY_LIMIT
tasks = []
for benchmark in benchmark_sets:
    if QUERY_LIMIT is not None and len(tasks) >= QUERY_LIMIT:
        break
    for query_index, query in enumerate(benchmark['queries'], start=1):
        if QUERY_LIMIT is not None and len(tasks) >= QUERY_LIMIT:
            break
        tasks.append((benchmark, query_index, query))

def _call_initial(args):
    benchmark, query_index, query = args
    prompt = build_prompt(benchmark['services'], query['query'], PROMPT_TEMPLATE)
    generated = call_llm(prompt, MODEL, '')
    generated += '\n\ncompose()'
    return {
        'query_index': query_index,
        'sector_name': benchmark['name'],
        'query': query,
        'prompt': prompt,
        'generated': generated,
        'service_files': benchmark.get('service_files', []),
        'model': MODEL
    }

sector_results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(_call_initial, t): t for t in tasks}
    for future in as_completed(futures):
        result = future.result()
        sector_results.append(result)
        print(f"[{len(sector_results)}/{total_queries}] [{result['sector_name']}] Query {result['query_index']} done")

sector_results.sort(key=lambda r: (r['sector_name'], r['query_index']))

[1/50] [01-energy] Query 7 done
[2/50] [01-energy] Query 8 done
[3/50] [01-energy] Query 2 done
[4/50] [01-energy] Query 1 done
[5/50] [01-energy] Query 6 done
[6/50] [01-energy] Query 10 done
[7/50] [01-energy] Query 4 done
[8/50] [01-energy] Query 9 done
[9/50] [01-energy] Query 5 done
[10/50] [02-materials] Query 1 done
[11/50] [02-materials] Query 6 done
[12/50] [01-energy] Query 3 done
[13/50] [02-materials] Query 4 done
[14/50] [02-materials] Query 3 done
[15/50] [02-materials] Query 7 done
[16/50] [02-materials] Query 5 done
[17/50] [02-materials] Query 2 done
[18/50] [02-materials] Query 10 done
[19/50] [03-industrials] Query 2 done
[20/50] [03-industrials] Query 3 done
[21/50] [02-materials] Query 8 done
[22/50] [03-industrials] Query 5 done
[23/50] [03-industrials] Query 1 done
[24/50] [03-industrials] Query 7 done
[25/50] [02-materials] Query 9 done
[26/50] [03-industrials] Query 6 done
[27/50] [03-industrials] Query 4 done
[28/50] [04-consumer discretionary] Query 6 done
[2

In [17]:
# Evaluate the original generated code for every query in the selected sector
if benchmark_sets:
    for result in sector_results:
        initial_metrics = evaluate(result['generated'], result['query'].get('endpoints', []))
        result['initial_metrics'] = initial_metrics
        print_metrics(initial_metrics, f"Initial evaluation - Query {result['query_index']}")


Initial evaluation - Query 1
  Precision: 0.80
  Recall:    0.80
  F1:        0.80
  Extracted: ['GET /alerts', 'GET /equipment-monitoring', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Expected:  ['GET /alerts', 'GET /equipment-status', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Missing:   ['GET /equipment-status']
  Extra:     ['GET /equipment-monitoring']
Initial evaluation - Query 2
  Precision: 1.00
  Recall:    1.00
  F1:        1.00
  Extracted: ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Expected:  ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Missing:   []
  Extra:     []
Initial evaluation - Query 3
  Precision: 0.75
  Recall:    0.75
  F1:        0.75
  Extracted: ['GET /electricity-demand', 'GET /ener

In [18]:
# Analyze the generated code with a Python linter helper for every query in the selected sector
if benchmark_sets:
    for result in sector_results:
        analysis = analyze(result['generated'])
        result['analysis'] = analysis
        print(f"Analysis - Query {result['query_index']}\n", analysis)


Analysis - Query 1
 AST: No syntax errors found.
Ruff: F841 Local variable `equipment_data` is assigned to but never used
Ruff:  --> /var/folders/57/p3z5bqsd11l937yr1409c3m00000gn/T/tmp9jvuv0p6/generated.py:6:5
Ruff:   |
Ruff: 4 |     # Retrieve the status and performance metrics of all monitored energy equipment
Ruff: 5 |     equipment_response = requests.get("https://api.energyinsights.com/v1/equipment-monitoring")
Ruff: 6 |     equipment_data = equipment_response.json()
Ruff:   |     ^^^^^^^^^^^^^^
Ruff: 7 |
Ruff: 8 |     # Access active alerts for any system performance issues
Ruff:   |
Ruff: help: Remove assignment to unused variable `equipment_data`
Ruff: F841 Local variable `alerts_data` is assigned to but never used
Ruff:   --> /var/folders/57/p3z5bqsd11l937yr1409c3m00000gn/T/tmp9jvuv0p6/generated.py:10:5
Ruff:    |
Ruff:  8 |     # Access active alerts for any system performance issues
Ruff:  9 |     alerts_response = requests.get("https://api.example.com/v1/alerts")
Ruff: 10 

In [19]:
# Run the refinement LLM call in parallel for every query
# Skipped entirely in BASELINE mode
if benchmark_sets and not BASELINE:
    def _call_refined(result):
        original_code = result['generated'].removesuffix('\n\ncompose()')
        refined_prompt = f'''{result['prompt']}

You are given a review of the code you generated and the code itself.
Use the review to improve the code.

## Analysis
{result['analysis']}

## Original Code
{original_code}

'''
        generated_refined = call_llm(refined_prompt, MODEL, 'Return only Python code, no explanation.')
        generated_refined += '\n\ncompose()'
        result['generated_refined'] = generated_refined
        return result

    refined_count = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_call_refined, r): r for r in sector_results}
        for future in as_completed(futures):
            future.result()  # result is mutated in-place
            refined_count += 1
            print(f"[{refined_count}/{total_queries}] Refinement done")
else:
    print("[BASELINE] Skipping refinement.")

[BASELINE] Skipping refinement.


In [20]:
# Final evaluation of the refined output for every query in the selected sector
# Skipped in BASELINE mode
if not BASELINE:
    for result in sector_results:
        refined_metrics = evaluate(result['generated_refined'], result['query'].get('endpoints', []))
        result['refined_metrics'] = refined_metrics
        print_metrics(refined_metrics, f"Refined evaluation - [{result['sector_name']}] Query {result['query_index']}")
else:
    print("[BASELINE] No refined evaluation.")

[BASELINE] No refined evaluation.


In [21]:
# Save outputs to disk under output/<date>_<mode>
from datetime import datetime
mode_tag = "baseline" if BASELINE else "refined"
run_name = datetime.now().strftime('%Y-%m-%d_%H-%M-%S') + f"_{mode_tag}"
# ensure we're using the latest output module implementation
importlib.reload(output_module)
outdir = output_module.make_output_dir(run_name)
for r in sector_results:
    output_module.write_query_output(outdir, r)

run_config = {
    'benchmark_dir': str(BENCHMARK_DIR),
    'benchmark_types': BENCHMARK_TYPES,
    'benchmark_limit': BENCHMARK_LIMIT,
    'query_limit': QUERY_LIMIT,
    'baseline': BASELINE,
    'model': MODEL,
}
output_module.write_overall_summary(outdir, sector_results, run_config)
print('Wrote outputs to', outdir)

Wrote outputs to output/2026-06-15_16-30-14_baseline


In [22]:
print_evaluation_summary([result['initial_metrics'] for result in sector_results], "Initial Evaluation Summary")
if not BASELINE:
    print_evaluation_summary([result['refined_metrics'] for result in sector_results], "Refined Evaluation Summary")

Initial Evaluation Summary
  Average Precision: 0.56
  Average Recall:    0.62
  Average F1:        0.58
  Avg. Missing Endpoints: 1.68
  Avg. Extra Endpoints:   2.20
  Correct Compositions: 8/50 (16.0%)
